# 05 — Execução da Camada de Consumo

Constrói os produtos finais de Consumo a partir dos arquivos da camada Analytic.

A execução é delegada à função `executar_camada_consumo()` do módulo `enem_pipeline.consumo`. Este notebook configura, valida as entradas, executa o processamento e apresenta a auditoria, sem reproduzir a lógica da camada.

## 1. Configuração e importações

Defina a UF de processamento. Os diretórios e parâmetros do projeto são obtidos do módulo de configuração.

In [1]:
from _setup import *

from pathlib import Path
import pandas as pd
from IPython.display import display

from enem_pipeline.config import (
    PROJECT_ROOT,
    ANALYTIC_DIR,
    CONSUMO_DIR,
    METADATA_DIR,
    normalizar_uf,
)
from enem_pipeline.consumo import executar_camada_consumo

UF_EXECUCAO = "RR"
UF_EXECUCAO = normalizar_uf(UF_EXECUCAO)

DIRETORIO_ANALYTIC_UF = ANALYTIC_DIR / f"uf={UF_EXECUCAO}"
DIRETORIO_CONSUMO_UF = CONSUMO_DIR / f"uf={UF_EXECUCAO}"

MANIFESTO_ANALYTIC = METADATA_DIR / f"manifesto_analytic_{UF_EXECUCAO}.csv"
MANIFESTO_CONSUMO = METADATA_DIR / f"manifesto_consumo_{UF_EXECUCAO}.csv"
CATALOGO_CONSUMO = METADATA_DIR / f"catalogo_consumo_{UF_EXECUCAO}.csv"
AUDITORIA_CONSUMO = METADATA_DIR / f"auditoria_consumo_{UF_EXECUCAO}.csv"

print("Projeto:", PROJECT_ROOT)
print("UF:", UF_EXECUCAO)
print("Diretório Analytic:", DIRETORIO_ANALYTIC_UF)
print("Manifesto Analytic:", MANIFESTO_ANALYTIC)
print("Diretório Consumo:", DIRETORIO_CONSUMO_UF)

Projeto: /home/akel/PycharmProjects/Projeto_ENEM
UF: RR
Diretório Analytic: /home/akel/PycharmProjects/Projeto_ENEM/data/analytic/uf=RR
Manifesto Analytic: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_analytic_RR.csv
Diretório Consumo: /home/akel/PycharmProjects/Projeto_ENEM/data/consumo/uf=RR


## 2. Validação das entradas

Confere a existência do manifesto Analytic, os produtos exigidos pela camada de Consumo e os respectivos arquivos Parquet.

A camada exige seis produtos para compor o painel municipal e três produtos auxiliares.

In [2]:
if not MANIFESTO_ANALYTIC.is_file():
    raise FileNotFoundError(
        f"Manifesto Analytic não encontrado: {MANIFESTO_ANALYTIC}"
    )

manifesto_analytic = pd.read_csv(
    MANIFESTO_ANALYTIC,
    sep=";",
    dtype={"uf": "string"},
)

colunas_obrigatorias = {"produto", "arquivo_analytic"}
ausentes = colunas_obrigatorias - set(manifesto_analytic.columns)
if ausentes:
    raise ValueError(
        f"Colunas ausentes no manifesto Analytic: {sorted(ausentes)}"
    )

if manifesto_analytic["produto"].duplicated().any():
    raise ValueError("O manifesto Analytic contém produtos duplicados.")

produtos_painel = {
    "demografia_municipal",
    "presenca_municipal",
    "desempenho_redacao_municipal",
    "desempenho_objetiva_municipal",
    "desempenho_areas_municipal",
    "escola_municipal",
}
produtos_auxiliares = {
    "faixa_etaria_municipal",
    "desempenho_grupos_municipal",
    "desempenho_sexo_faixa_etaria_municipal",
}
produtos_esperados = produtos_painel | produtos_auxiliares
produtos_encontrados = set(manifesto_analytic["produto"].dropna())

produtos_ausentes = produtos_esperados - produtos_encontrados
if produtos_ausentes:
    raise ValueError(
        f"Produtos Analytic ausentes: {sorted(produtos_ausentes)}"
    )

arquivos_ausentes = [
    str(caminho)
    for caminho in manifesto_analytic["arquivo_analytic"].dropna()
    if not Path(str(caminho)).is_file()
]
if arquivos_ausentes:
    raise FileNotFoundError(
        "Arquivos Analytic ausentes:\n" + "\n".join(arquivos_ausentes)
    )

if "uf" in manifesto_analytic.columns:
    ufs_manifesto = set(
        manifesto_analytic["uf"].dropna().astype(str).str.upper()
    )
    if ufs_manifesto and ufs_manifesto != {UF_EXECUCAO}:
        raise ValueError(
            f"UFs no manifesto ({sorted(ufs_manifesto)}) "
            f"diferem da UF selecionada ({UF_EXECUCAO})."
        )

print("Produtos Analytic exigidos:", len(produtos_esperados))
print("Produtos encontrados:", len(produtos_encontrados & produtos_esperados))
print("Arquivos Analytic verificados:", len(manifesto_analytic))
display(
    manifesto_analytic[
        [c for c in ["produto", "arquivo_analytic"] if c in manifesto_analytic.columns]
    ]
)

Produtos Analytic exigidos: 9
Produtos encontrados: 9
Arquivos Analytic verificados: 9


,produto,arquivo_analytic
0,demografia_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
1,faixa_etaria_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
2,presenca_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
3,desempenho_redacao_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
4,desempenho_objetiva_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
5,desempenho_areas_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
6,escola_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
7,desempenho_grupos_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...
8,desempenho_sexo_faixa_etaria_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...


## 3. Execução da camada de Consumo

Com as entradas validadas, executa a função existente no módulo `consumo.py`.

A função gera quatro produtos Parquet, além do catálogo, manifesto e arquivo de auditoria.

In [3]:
resultado_consumo = executar_camada_consumo(
    manifesto_analytic_path=MANIFESTO_ANALYTIC,
    metadata_dir=METADATA_DIR,
    consumo_root=CONSUMO_DIR,
    uf=UF_EXECUCAO,
    gravar=True,
)

print(f"Tempo de execução: {resultado_consumo['duracao_segundos']:.2f} s")
print("Diretório de saída:", resultado_consumo["diretorio_consumo"])

Tempo de execução: 0.30 s
Diretório de saída: /home/akel/PycharmProjects/Projeto_ENEM/data/consumo/uf=RR


## 4. Inspeção dos produtos e da auditoria

Apresenta o catálogo, o manifesto produzido e os testes executados pela própria camada. Também verifica se os quatro arquivos Parquet foram gravados.

In [4]:
catalogo_consumo = resultado_consumo["catalogo"]
manifesto_consumo = resultado_consumo["manifesto"]
auditoria_consumo = resultado_consumo["auditoria"]

display(catalogo_consumo)
display(manifesto_consumo)
display(auditoria_consumo)

arquivos_consumo_ausentes = [
    str(caminho)
    for caminho in catalogo_consumo["arquivo_consumo"]
    if not Path(str(caminho)).is_file()
]

print("Produtos de Consumo:", len(catalogo_consumo))
print("Arquivos Parquet ausentes:", len(arquivos_consumo_ausentes))
if arquivos_consumo_ausentes:
    display(arquivos_consumo_ausentes)

print(
    "Resultado da auditoria:",
    "OK" if resultado_consumo["auditoria_ok"] else "COM FALHAS"
)

,produto,arquivo_consumo,granularidade,periodo,linhas,colunas,descricao
0,painel_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,ano_municipio_referencia,1998-2025,817,82,"Painel integrado de indicadores municipais, re..."
1,faixa_etaria_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,ano_municipio_referencia_faixa_etaria,1998-2025,5987,7,"Distribuição de registros por faixa etária, mu..."
2,desempenho_grupos_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,ano_municipio_referencia_grupo_disciplina,1998-2025,30182,16,"Estatísticas de notas por grupo, disciplina, m..."
3,desempenho_sexo_faixa_etaria_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,ano_municipio_referencia_sexo_faixa_etaria_dis...,2001-2023,38548,17,"Estatísticas de notas por sexo, faixa etária, ..."


,camada,uf,produto,granularidade,periodo,arquivo_consumo,n_linhas,n_colunas,tamanho_mb
0,consumo,RR,painel_municipal,ano_municipio_referencia,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,817,82,0.184011
1,consumo,RR,faixa_etaria_municipal,ano_municipio_referencia_faixa_etaria,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,5987,7,0.019205
2,consumo,RR,desempenho_grupos_municipal,ano_municipio_referencia_grupo_disciplina,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,30182,16,0.798902
3,consumo,RR,desempenho_sexo_faixa_etaria_municipal,ano_municipio_referencia_sexo_faixa_etaria_dis...,2001-2023,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,38548,17,0.852548


,teste,ok,obtido
0,produto_painel_municipal_nao_vazio,True,817
1,produto_faixa_etaria_municipal_nao_vazio,True,5987
2,produto_desempenho_grupos_municipal_nao_vazio,True,30182
3,produto_desempenho_sexo_faixa_etaria_municipal...,True,38548
4,chave_painel_unica,True,0
5,chave_faixa_unica,True,0
6,chave_grupos_unica,True,0
7,chave_sexo_faixa_unica,True,0
8,reconciliacao_escola,True,0


Produtos de Consumo: 4
Arquivos Parquet ausentes: 0
Resultado da auditoria: OK


## 5. Critérios de conclusão

A execução é considerada concluída quando:

- os quatro produtos de Consumo estão catalogados;
- os quatro arquivos Parquet existem no diretório da UF;
- o catálogo, o manifesto e a auditoria foram gravados;
- todos os testes registrados pela função de auditoria foram aprovados.

A auditoria de reconciliação das categorias de escola é incluída pelo módulo quando as colunas necessárias estão disponíveis.

In [5]:
produtos_consumo_esperados = {
    "painel_municipal",
    "faixa_etaria_municipal",
    "desempenho_grupos_municipal",
    "desempenho_sexo_faixa_etaria_municipal",
}

produtos_consumo_encontrados = set(catalogo_consumo["produto"])
produtos_consumo_ausentes = (
    produtos_consumo_esperados - produtos_consumo_encontrados
)

arquivos_metadata = [
    MANIFESTO_CONSUMO,
    CATALOGO_CONSUMO,
    AUDITORIA_CONSUMO,
]

metadata_ausente = [
    str(caminho) for caminho in arquivos_metadata if not caminho.is_file()
]

criterios = {
    "quatro_produtos_catalogados": (
        len(catalogo_consumo) == 4
        and not produtos_consumo_ausentes
    ),
    "quatro_parquets_existentes": (
        len(arquivos_consumo_ausentes) == 0
        and len(catalogo_consumo) == 4
    ),
    "arquivos_de_metadata_gravados": len(metadata_ausente) == 0,
    "auditoria_aprovada": bool(resultado_consumo["auditoria_ok"]),
}

for criterio, aprovado in criterios.items():
    print(f"{'OK' if aprovado else 'FALHOU'} — {criterio}")

if produtos_consumo_ausentes:
    print("Produtos ausentes:", sorted(produtos_consumo_ausentes))
if metadata_ausente:
    print("Arquivos de metadata ausentes:", metadata_ausente)

if not all(criterios.values()):
    raise AssertionError(
        "A camada de Consumo não atendeu a todos os critérios de conclusão."
    )

print("\nCAMADA DE CONSUMO CONCLUÍDA COM SUCESSO.")
print("UF:", UF_EXECUCAO)
print("Diretório:", DIRETORIO_CONSUMO_UF)

OK — quatro_produtos_catalogados
OK — quatro_parquets_existentes
OK — arquivos_de_metadata_gravados
OK — auditoria_aprovada

CAMADA DE CONSUMO CONCLUÍDA COM SUCESSO.
UF: RR
Diretório: /home/akel/PycharmProjects/Projeto_ENEM/data/consumo/uf=RR
